# Noise and Decoherence

## Real qubits are not the perfect mathematical objects of the previous chapters. They couple to their environment, drift, and decay. This chapter is about identifying that noise and then *correcting* it with quantum error correction. We start by understanding what the enemy looks like.

# 1. The two main culprits

## - **$T_1$ — relaxation / amplitude damping**: the qubit decays from $|1\rangle$ toward $|0\rangle$ on a timescale   $T_1$ (microseconds for superconducting qubits, milliseconds-to-seconds for trapped ions).
## - **$T_2$ — dephasing**: the relative phase between $|0\rangle$ and $|1\rangle$ randomises on a timescale $T_2$.   In practice $T_2 \leq 2 T_1$.

## Together these collapse a qubit's Bloch vector toward the $+z$ axis.

# 2. Pauli error channels

## A clean theoretical model: with some probability $p$, a random Pauli is applied to each qubit:

## - $X$ error — bit flip.
## - $Z$ error — phase flip.
## - $Y$ error — combined (since $Y = iXZ$).

## The **depolarising channel** applies each of $X$, $Y$, $Z$ with probability $p/3$, leaving the qubit alone with probability $1 - p$. It is the standard worst-case noise model in QEC.

# 3. Density matrices and mixed states

## A noisy quantum state is described by a **density matrix** $\rho$ — a positive, trace-1 Hermitian operator. A pure state is just $\rho = |\psi\rangle\langle\psi|$. A mixed state lives strictly inside the Bloch sphere.

## Channels are described by **completely positive trace-preserving (CPTP) maps**, often written in Kraus form:

$$ \Large  \mathcal{E}(\rho) = \sum_k K_k\, \rho\, K_k^\dagger, \qquad \sum_k K_k^\dagger K_k = I. $$

In [ ]:
# Simulate a single-qubit depolarising channel
import numpy as np

I = np.eye(2, dtype=complex)
X = np.array([[0,1],[1,0]], dtype=complex)
Y = np.array([[0,-1j],[1j,0]], dtype=complex)
Z = np.array([[1,0],[0,-1]], dtype=complex)

def depolarise(rho, p):
    return (1-p)*rho + (p/3)*(X@rho@X + Y@rho@Y + Z@rho@Z)

psi = np.array([1, 1], dtype=complex)/np.sqrt(2)   # |+>
rho = np.outer(psi, psi.conj())
for p in [0, 0.1, 0.3, 0.5, 0.75]:
    out = depolarise(rho, p)
    purity = np.trace(out @ out).real
    print(f'p = {p}, purity = {purity:.3f}')

# 4. Coherent vs incoherent errors

## - **Incoherent** errors look like random Pauli flips, statistically independent across runs. Easy to model and correct.
## - **Coherent** errors are systematic — e.g. a gate that consistently over-rotates by 1%. They are *worse* than they look because their effects can add coherently across many gates instead of averaging out.

## Randomised compiling (Wallman & Emerson) is a clever trick to convert coherent errors into incoherent ones in software.

# 5. The discretisation miracle

## Quantum error correction works because **detecting and correcting Pauli errors is enough**. The argument: any single-qubit error operator $E$ can be expanded as $E = \alpha I + \beta X + \gamma Y + \delta Z$, so the action of $E$ on an encoded state is a superposition of "do nothing", "$X$ error", etc. Measurement of the *syndrome* (next notebook) collapses this to a definite Pauli, which can then be corrected.

## This is what makes QEC tractable: we only need to deal with a discrete set of errors, not a continuum.

# Recap

## - $T_1$ (decay) and $T_2$ (dephasing) are the dominant hardware error sources.
## - Pauli error channels are the standard mathematical model.
## - Density matrices + CPTP maps describe noisy evolution.
## - QEC works because correcting Paulis is enough ("discretisation of errors").

## Next: our first error-correcting code — the 3-qubit bit-flip code.